# Data Validation: AIDSVu + ACS County-Year Panel (2013–2023)

This notebook validates the five preprocessing issues documented in the project report's Methodology section, against `data/aidsvu_acs_county_panel_2013_2023.csv`. Every check below is run directly against the file in this repo, so the results here are reproducible by anyone who clones it.

Project: MSBD 503 (Introduction to Epidemiology), reproducing and extending Garrett et al. (2026), *AIDS and Behavior*.
Author: Sade M. Graves


In [ ]:
import pandas as pd

df = pd.read_csv("../data/aidsvu_acs_county_panel_2013_2023.csv")
print("Shape:", df.shape)
print("Columns:", list(df.columns))


Shape: (35442, 19)
Columns: ['Year', 'GEO_ID', 'State', 'StateAbbr', 'CountyName', 'NewDX_Cases', 'NewDX_Rate', 'NewDX_RateStability', 'PovertyPct', 'UnempPct', 'UninsuredPct', 'HSGradPct', 'WhitePct', 'BlackPct', 'AsianPct', 'TotalPop', 'NoHSGradPct', 'OtherRacePct', 'HispanicPct']


## 1. Panel completeness

The panel should be a complete, balanced set of counties by year: 3,222 counties x 11 years.

In [ ]:
print("Unique counties:", df['GEO_ID'].nunique())
print("Years covered:", sorted(df['Year'].unique().tolist()))
print("Expected rows (counties x years):", df['GEO_ID'].nunique() * df['Year'].nunique())
print("Actual rows:", len(df))


Unique counties: 3222
Years covered: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Expected rows (counties x years): 35442
Actual rows: 35442


## 2. Embedded-newline check (Issue #1)

The original raw AIDSVu export had corrupted rows from embedded line breaks in the county-name field, which showed up as non-numeric or implausible `Year` values. Confirming none remain in the cleaned file.

In [ ]:
bad_years = df[~df['Year'].astype(str).str.fullmatch(r'20(1[3-9]|2[0-3])')]
print("Rows with an implausible Year value:", len(bad_years))


Rows with an implausible Year value: 0


## 3. Census API missing-data sentinel check (Issue #4)

The Census API sometimes encodes an unavailable estimate as a large negative placeholder (e.g. -888,888,888) instead of leaving the cell blank. Checking that no such sentinel remains in any numeric covariate.

In [ ]:
numeric_cols = ['PovertyPct','UnempPct','UninsuredPct','HSGradPct','WhitePct','BlackPct',
                'AsianPct','HispanicPct','NoHSGradPct','OtherRacePct']
flagged = {c: int((df[c].abs() > 1000).sum()) for c in numeric_cols if (df[c].abs() > 1000).sum() > 0}
print("Columns with remaining extreme-magnitude (sentinel-like) values:", flagged if flagged else "none")

rio = df[(df['GEO_ID'] == 35039) & (df['Year'] == 2018)]
print("\nRio Arriba County, NM, 2018 (the one row that carried the sentinel):")
print(rio[['CountyName','Year','PovertyPct','UnempPct']].to_string(index=False))


Columns with remaining extreme-magnitude (sentinel-like) values: none

Rio Arriba County, NM, 2018 (the one row that carried the sentinel):
       CountyName  Year  PovertyPct  UnempPct
Rio Arriba County  2018         NaN       NaN


## 4. AIDSVu suppression-code check in the outcome variable (Issue #5)

AIDSVu marks counties with low case counts using negative placeholder codes (-1, -4, -9) in `NewDX_Rate` and `NewDX_Cases`, to avoid risking re-identification. These must be treated as missing, not as literal negative rates. Confirming none remain as negative numbers after cleaning.

In [ ]:
neg_rate = int((df['NewDX_Rate'] < 0).sum())
neg_cases = int((df['NewDX_Cases'] < 0).sum())
print("Negative NewDX_Rate values remaining:", neg_rate)
print("Negative NewDX_Cases values remaining:", neg_cases)
print("NewDX_Rate missing (suppressed) count:", df['NewDX_Rate'].isna().sum(), "of", len(df),
      f"({df['NewDX_Rate'].isna().mean()*100:.1f}%)")
print("\nOverall mean NewDX_Rate (suppressed rows excluded):", round(df['NewDX_Rate'].mean(), 2))


Negative NewDX_Rate values remaining: 0
Negative NewDX_Cases values remaining: 0
NewDX_Rate missing (suppressed) count: 12786 of 35442 (36.1%)

Overall mean NewDX_Rate (suppressed rows excluded): 5.89


## 5. Table 1 reproduction: regional summary statistics

Reproduces the exact numbers in the report's Table 1, computed straight from this file, broken out by U.S. Census region.

In [ ]:
region_map = {
 'CT':'North','ME':'North','MA':'North','NH':'North','RI':'North','VT':'North','NJ':'North','NY':'North','PA':'North',
 'IL':'Midwest','IN':'Midwest','MI':'Midwest','OH':'Midwest','WI':'Midwest','IA':'Midwest','KS':'Midwest','MN':'Midwest',
 'MO':'Midwest','NE':'Midwest','ND':'Midwest','SD':'Midwest',
 'DE':'South','FL':'South','GA':'South','MD':'South','NC':'South','SC':'South','VA':'South','DC':'South','WV':'South',
 'AL':'South','KY':'South','MS':'South','TN':'South','AR':'South','LA':'South','OK':'South','TX':'South','PR':'South',
 'AZ':'West','CO':'West','ID':'West','MT':'West','NV':'West','NM':'West','UT':'West','WY':'West','AK':'West','CA':'West',
 'HI':'West','OR':'West','WA':'West',
}
df['Region'] = df['StateAbbr'].map(region_map)

cols = ['PovertyPct','UnempPct','UninsuredPct','NoHSGradPct','BlackPct','WhitePct','AsianPct','HispanicPct','NewDX_Rate']
rows = []
for reg in ['Overall','North','South','West','Midwest']:
    sub = df if reg == 'Overall' else df[df['Region'] == reg]
    row = {'Region': reg, 'Counties': sub['GEO_ID'].nunique()}
    for c in cols:
        row[c] = round(sub[c].mean(), 1)
    rows.append(row)

summary = pd.DataFrame(rows).set_index('Region')
print(summary.to_string())


         Counties  PovertyPct  UnempPct  UninsuredPct  NoHSGradPct  BlackPct  WhitePct  AsianPct  HispanicPct  NewDX_Rate
Region                                                                                                                   
Overall      3222        16.3       6.6          11.2         13.4       9.0      81.5       1.3         11.5         5.9
North         218        12.2       6.2           7.0          9.9       5.2      86.5       2.6          6.7         6.5
South        1500        19.7       7.8          13.1         16.9      16.3      74.6       1.0         15.2        10.4
West          449        14.5       6.7          12.2         11.2       1.3      80.8       2.6         17.3         3.1
Midwest      1055        13.1       5.1           8.8         10.2       2.5      90.7       0.9          4.6         1.7


## Summary

All five preprocessing issues described in the Methodology section are confirmed fixed in this file: no corrupted rows, no remaining Census API sentinel values, no remaining negative AIDSVu suppression codes in the outcome variable, and the regional summary statistics above match Table 1 in the report. The corrected overall new-diagnosis rate (5.9 per 100,000) is close to the original study's own reported regional rates, which is an independent check that the cleaning was done correctly rather than an arbitrary adjustment.